In [ ]:
# ===============================
# 🧠 Animal Detection using 64-block labels per image
# ===============================

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)
from tqdm import tqdm

# ===============================
# 1️⃣ Load Excel File
# ===============================
# Expected format:
# image_name | block_0 | block_1 | ... | block_63
df = pd.read_excel(r"C:\Users\Dell\Desktop\IIT\ds203\PROJECT\manual_classification.xlsx")
print("Data sample:")
print(df.head())

IMAGE_FOLDER = r"C:\Users\Dell\Desktop\IIT\ds203\PROJECT\project_images\images_resized"  # ⬅️ Update this path to your resized image directory

# ===============================
# 2️⃣ Feature Extraction Functions
# ===============================
def extract_hsv_features(image):
    """Extract HSV color histogram features."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8],
                        [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

# --- Optional LBP feature (commented) ---
from skimage.feature import local_binary_pattern
def extract_lbp_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 10), range=(0, 9))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    return hist

# --- Optional Laplacian feature (commented) ---
def extract_laplacian_features(image):
 gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
 lap = cv2.Laplacian(gray, cv2.CV_64F)
 return np.array([lap.var()])

# ===============================
# 3️⃣ Split images into 64 squares & extract features
# ===============================
X, y = [], []

for _, row in tqdm(df.iterrows(), total=len(df)):
    img_name = row["filename"]
    img_path = os.path.join(IMAGE_FOLDER, img_name)

    if not os.path.exists(img_path):
        print(f"⚠️ Missing image: {img_path}")
        continue

    image = cv2.imread(img_path)
    if image is None:
        continue

    h, w, _ = image.shape
    grid_rows, grid_cols = 8, 8
    patch_h, patch_w = h // grid_rows, w // grid_cols

    # Extract features for each block
    for block_id in range(64):
        i, j = divmod(block_id, 8)
        y1, y2 = i * patch_h, (i + 1) * patch_h
        x1, x2 = j * patch_w, (j + 1) * patch_w
        patch = image[y1:y2, x1:x2]

        feats = extract_hsv_features(patch)
        lbp_feats = extract_lbp_features(patch)
        lap_feats = extract_laplacian_features(patch)
        feats = np.concatenate([feats, lbp_feats, lap_feats])

        X.append(feats)
        y.append(int(row[f"block_{block_id}"]))

X = np.array(X)
y = np.array(y)

print("✅ Feature matrix shape:", X.shape)
print("✅ Label count:", len(y))

# ===============================
# 4️⃣ Train/Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ===============================
# 5️⃣ Scale & Train SVM
# ===============================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC(kernel="rbf", probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

# ===============================
# 6️⃣ Evaluate Model
# ===============================
y_pred = svm.predict(X_test_scaled)
y_prob = svm.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ===============================
# 7️⃣ Visualization of Predictions (20 Random Images)
# ===============================








In [ ]:
# ===============================
# 8️⃣ ROC Curve with Variance (Bootstrap Confidence Interval)
# ===============================
from sklearn.metrics import roc_curve, auc
from sklearn.utils import resample

plt.figure(figsize=(8, 6))

# --- Compute base ROC ---
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='blue', lw=2, label=f"ROC (AUC = {roc_auc:.3f})")

# --- Bootstrap for Variance Estimation ---
n_bootstraps = 100  # increase for smoother CI, e.g., 500
rng = np.random.RandomState(42)
boot_tprs = []
mean_fpr = np.linspace(0, 1, 100)

for i in range(n_bootstraps):
    indices = rng.randint(0, len(y_test), len(y_test))
    if len(np.unique(y_test[indices])) < 2:
        continue  # skip if resample lacks both classes
    y_prob_boot = y_prob[indices]
    y_test_boot = y_test[indices]
    fpr_boot, tpr_boot, _ = roc_curve(y_test_boot, y_prob_boot)
    boot_tpr = np.interp(mean_fpr, fpr_boot, tpr_boot)
    boot_tprs.append(boot_tpr)

# --- Compute Confidence Interval (95%) ---
boot_tprs = np.array(boot_tprs)
tpr_mean = boot_tprs.mean(axis=0)
tpr_std = boot_tprs.std(axis=0)
tpr_upper = np.minimum(tpr_mean + 1.96 * tpr_std, 1)
tpr_lower = np.maximum(tpr_mean - 1.96 * tpr_std, 0)

plt.plot(mean_fpr, tpr_mean, color='orange', lw=2, linestyle='--', label="Mean ROC (bootstrap)")
plt.fill_between(mean_fpr, tpr_lower, tpr_upper, color='orange', alpha=0.3, label="95% CI")

# --- Plot settings ---
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve with 95% Confidence Interval")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

def visualize_predictions(img_path, model, scaler, threshold=0.5):
    image = cv2.imread(img_path)
    if image is None:
        return None

    h, w, _ = image.shape
    ph, pw = h // 8, w // 8
    overlay = image.copy()

    for i in range(8):
        for j in range(8):
            y1, y2 = i * ph, (i + 1) * ph
            x1, x2 = j * pw, (j + 1) * pw
            patch = image[y1:y2, x1:x2]

            # --- Extract all features used during training ---
            hsv_feats = extract_hsv_features(patch)
            lbp_feats = extract_lbp_features(patch)
            lap_feats = extract_laplacian_features(patch)
            feats = np.concatenate([hsv_feats, lbp_feats, lap_feats])

            # --- Predict ---
            feats_scaled = scaler.transform([feats])
            prob = model.predict_proba(feats_scaled)[0, 1]

            if prob > threshold:
                # Full filled transparent red overlay
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 255), thickness=3)
            else:
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (255, 255, 255), thickness=1)

    # Optional: blend with original image for visibility
    output = cv2.addWeighted(image, 0.6, overlay, 0.4, 0)
    return cv2.cvtColor(output, cv2.COLOR_BGR2RGB)

# --- Show 20 random images separately ---
sample_images = df["filename"].sample(20, random_state=random.randint(0, 9999)).tolist()

for idx, filename in enumerate(sample_images, 1):
    img_path = os.path.join(IMAGE_FOLDER, filename)
    vis = visualize_predictions(img_path, svm, scaler, threshold=0.5)
    if vis is not None:
        plt.figure(figsize=(6, 6))
        plt.imshow(vis)
        plt.axis("off")
        plt.title(f"{filename}")
        plt.show()
